# Alert Stream and Visits 

Query the number of diaSources reported in `lsst.prompt.prod.numDiaSourcesGood` and merge with visit information.

In [ ]:
import os
from astropy.time import Time, TimeDelta
from IPython.display import display, HTML
from rubin_nights import connections
from rubin_nights.influx_query import InfluxQueryClient
import rubin_nights.dayobs_utils as rn_dayobs

import numpy as np
import pandas as pd 

import matplotlib.pyplot as plt

import logging
logging.getLogger('rubin_nights').setLevel(logging.INFO)

In [ ]:
# Connect using your tokenfile and appropriate site 
tokenfile = "/Users/lynnej/.lsst/usdf_rsp"
site = "usdf"
endpoints = connections.get_clients(tokenfile, site)

In [ ]:
day_obs = 20250915

# Find start and end of the night
tstart, tend = rn_dayobs.day_obs_sunset_sunrise(day_obs, sun_alt=-12)

sasquatch = InfluxQueryClient("usdfdev", db_name="lsst.prompt")
tt = sasquatch.select_time_series("lsst.prompt.prod.numDiaSourcesGood", 
                                  ["visit", "band", "day_obs", "detector", "dataset_tag", "numAllDiaSources", "numGoodDiaSources", "run"], 
                                  tstart, tend)
if len(tt) > 0:
    alertsum = tt.groupby("visit").agg({'numAllDiaSources': 'sum', 'numGoodDiaSources': 'sum', 'detector': 'count'})
    alertsum.index = alertsum.index.astype(int)
    alertsum.rename({"detector": "nDiaDetectors"}, inplace=True, axis=1)
    tt.numGoodDiaSources.sum(), tt.numAllDiaSources.sum()
else:
    print("no alerts")

In [ ]:
visits = endpoints['consdb'].get_visits("lsstcam", tstart, tend, visit_constraint="science_program = 'BLOCK-365'")

In [ ]:
print("number of block-365 visits:", len(visits))
print("number of visits with alerts:", len(alertsum))

In [ ]:
if len(visits) > 0 and len(alertsum) > 0:
    vv = pd.merge(visits, alertsum, left_on='visit_id', right_index=True, how='outer')
    cols = ['visit_id', 'observation_reason', 'obs_start', 'band', 's_ra', 's_dec', 'sky_rotation', 'clouds', 
        'fwhm_eff', 'cat_m5', 'numGoodDiaSources', 'nDiaDetectors']
    vv[cols]

In [ ]:
if len(vv) > 0:
    plt.figure(figsize=(8, 8))
    for tt in ['ddf', 'pair']:
        qq = vv.query("observation_reason.str.contains(@tt)")
        if tt == 'ddf':
            marker='s'
        else:
            marker = 'o'
        for b in qq.band.unique():
            q = qq.query("band == @b")
            plt.scatter(q.cat_m5, q.numGoodDiaSources, s=q.nDiaDetectors/189 * 50, marker=marker, linestyle='', label=f"{b} {tt}")
            #plt.scatter(q.nDiaDetectors, q.numGoodDiaSources, s=q.cat_m5 * 2, marker=marker, linestyle='', label=f"{b} {tt}")
    plt.legend()
    plt.xlabel("estimated limiting magnitude", fontsize='large')
    #plt.xlabel("nDiaDetectors per visit")
    plt.ylabel("nGoodDiaSources per visit", fontsize='large')
    plt.title(f"Dayobs {day_obs}", fontsize='large')